# LangGraph 상태 관리: Reducer 와 MessagesState

## Reducer 와 MessagesState 는 무엇이고, 왜 필요한가

### Reducer: 노드의 결과를 상태에 "어떻게 합칠지" 정하는 규칙
* **역할**: 노드는 상태 전체가 아니라 **바꿀 키의 값만** 반환합니다. Reducer 는 그 반환값을 기존 상태 값과 **어떻게 합칠지**를 키별로 정의합니다. `함수(기존 값, 새 값) → 저장될 값` 형태입니다.
* **왜 필요한가**
    * Reducer 를 지정하지 않으면 새 값이 기존 값을 **덮어씁니다.** 검색 결과나 대화 기록처럼 **쌓여야 하는 데이터**는 노드를 거칠 때마다 이전 내용이 사라집니다.
    * 여러 노드가 **병렬로 같은 키**를 갱신하면, 합치는 규칙이 없을 때 충돌합니다. Reducer 가 있으면 결과를 정해진 방식으로 모읍니다.
* **지정 방법**: `Annotated[타입, reducer]` — 예) 이어 붙이기 `operator.add`, 중복 제거 같은 **직접 만든 함수**.

| 상황 | 상태 선언 | 노드가 `["b"]` 를 반환했을 때 (기존 값 `["a"]`) |
|---|---|---|
| Reducer 없음 | `docs: List[str]` | `["b"]` (덮어쓰기) |
| 이어 붙이기 | `docs: Annotated[List[str], add]` | `["a", "b"]` (누적) |
| 직접 만든 reducer | `docs: Annotated[List[str], my_reducer]` | `my_reducer(["a"], ["b"])` 의 결과 |

### MessagesState: 대화(메시지 목록)용으로 미리 만들어 둔 상태
* **역할**: `messages` 키 하나와 그 키의 reducer(`add_messages`)가 **이미 설정된 상태 클래스**입니다. 즉 `MessagesState` 는 "메시지 목록에 Reducer 를 적용한 상태"입니다.
    ```python
    # MessagesState 는 사실상 아래와 같은 상태입니다.
    class MessagesState(TypedDict):
        messages: Annotated[list[AnyMessage], add_messages]
    ```
* **왜 필요한가**
    * 대화형 앱은 `HumanMessage`, `AIMessage`, `ToolMessage` 가 **계속 쌓이는** 구조입니다. 노드는 새 메시지만 반환하면 되고, 이전 대화가 유지됩니다.
    * `add_messages` 는 단순 이어 붙이기(`add`)와 달리 **메시지 ID 로 기존 메시지를 갱신**하고, `("user", "안녕")` 같은 튜플도 메시지 객체로 변환해 줍니다.
    * 대화형 앱마다 같은 상태 정의를 반복해 쓰지 않아도 되고, 이후 노트북의 도구 호출(`ToolNode`)·에이전트(`create_agent`)가 모두 이 형식을 전제로 동작합니다.
    * 필드를 더하고 싶으면 `MessagesState` 를 **상속**해 추가합니다.
    * 주의: `MessagesState` 가 reducer 를 제공하는 키는 **`messages` 하나뿐**입니다. 상속해서 추가한 키는 reducer 를 지정하지 않으면 **덮어쓰기**입니다. 누적이 필요하면 `Annotated[타입, operator.add]` 처럼 기존 함수를 지정하고, 중복 제거 같은 특별한 규칙이 필요할 때만 커스텀 reducer 를 직접 작성합니다.
* **`MessageGraph` 와의 관계**: 이전 버전에서는 메시지 전용 그래프인 `MessageGraph` 를 사용했지만, 지금은 **`StateGraph(MessagesState)`** 를 사용합니다.

## 이 노트북의 역할
노드가 반환한 값이 **그래프 상태에 어떻게 합쳐지는지(Reducer)** 를 익히고, 대화형 앱에서 쓰는 **`MessagesState` / `add_messages`** 로 이어집니다.

| 단계 | 내용 | 배우는 것 |
|---|---|---|
| 1. State Reducer | (1) 기본(덮어쓰기) → (2) `operator.add` 로 누적 → (3) 커스텀 reducer(중복 제거) | `Annotated[타입, reducer]` |
| 2. Messages State | 메시지 목록을 상태로 관리, 멀티턴 대화, 상태 필드 확장 | `add_messages`, `MessagesState` |

## 핵심 규칙
* 노드는 상태 **전체가 아니라 바꿀 키만** 반환합니다. 반환값은 reducer 로 기존 값과 합쳐집니다.
* reducer 가 없으면 **덮어쓰기**, `Annotated[타입, 함수]` 로 지정하면 `함수(기존값, 새값)` 결과가 저장됩니다.
* 1단계(문서 목록)는 **LLM 없이** 실행됩니다. 2단계에서만 LLM 을 호출합니다.

## 사용 모델 (`get_llm()`)
- `PROVIDER` 값(`"groq"` / `"openai"` / `"upstage"`, 기본값 `"upstage"`)만 바꾸면 이후 LLM 이 바뀝니다. 바꾼 뒤에는 LLM 설정 셀부터 다시 실행하세요.
- LLM 호출은 사용 요금이 발생할 수 있습니다.

## 실행 시 주의
* 같은 이름의 변수(`builder`, `graph`, `node_1` ...)를 단계마다 다시 정의합니다. 셀을 **위에서 아래로 순서대로** 실행하세요.

### 1. State Reducer
- Reducer는 LangGraph에서 상태 업데이트를 관리하는 중요한 개념
- 그래프의 각 노드의 출력을 그래프의 상태에 통합하는 방법을 정의
- Reducer의 필요성
    - 상태 덮어쓰기 문제: 기본적으로 각 노드의 반환값은 해당 상태 키의 이전 값을 덮어쓰는 방식으로 동작 (override)
    - 누적 업데이트 필요: 특히 메시지 리스트와 같은 경우, 이전 상태에 새로운 값을 추가하고 싶을 때가 있음 

`(1) Reducer를 별도로 지정하지 않은 경우 `
- reducer를 별도로 지정하지 않은 경우 기존 값을 덮어쓰는 방식으로 동작
- 기본 reducer는 상태에 대해 별도의 설정 없이 사용될 때 자동으로 적용

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END


# 상태 정의: reducer 지정이 없으므로 모든 키는 "덮어쓰기"로 갱신됨
class DocumentState(TypedDict):
    query: str
    documents: List[str]


def node_1(state: DocumentState) -> DocumentState:
    """query 를 그대로 반환 (documents 는 건드리지 않음)"""
    print("---Node 1 (query update)---")
    query = state["query"]
    return {"query": query}


def node_2(state: DocumentState) -> DocumentState:
    """검색된 문서 1차 결과를 documents 에 저장"""
    print("---Node 2 (add documents)---")
    return {"documents": ["doc1.pdf", "doc2.pdf", "doc3.pdf"]}


def node_3(state: DocumentState) -> DocumentState:
    """2차 결과를 반환 — reducer 가 없으므로 node_2 의 결과를 덮어씀"""
    print("---Node 3 (add more documents)---")
    return {"documents": ["doc2.pdf", "doc4.pdf", "doc5.pdf"]}


# 그래프 빌드: START → node_1 → node_2 → node_3 → END
builder = StateGraph(DocumentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()

In [ ]:
# 그래프 구조 확인: 이미지 대신 mermaid 코드를 출력 (https://mermaid.live/ 에 붙여 넣으면 그림으로 보임)
mermaid_code = graph.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

In [ ]:
# 이미지로 직접 보려면: 
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

* https://mermaid.live/ 에서  mermaid_code 로 직접 확인한다.

* [Graph이미지](https://www.mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNp9j0EKgzAQRa8S7EahtppAF1FceYTuapGoGQ0EFU0XpfTujYkkuOlqZt78-T_5BO3U8YAG_cLmAd3LrBorVderYosu4SOfCzfl17l4RpRSEMuqNuGoj-s0tCVyBFuCPSGWkMja87Fz5qZ31pJZZxeK4rhANiDzkZ5iT7GnxFNi6J5jcKtD1pID6jiwl1QIhJT0BBgSgLMUI48HLvpB0fSCDwfm40YeTzNrhXrT5CDYnr_bNdDcoNXbv_vg-wPTVIcO)

In [ ]:
# 질문
initial_state = {"query": "Reducer를 사용하지 않은 경우"}

# 그래프 실행 
final_state = graph.invoke(initial_state)

# 최종 상태 출력
print("최종 상태:", final_state)

> **결과 해석**: `documents` 는 node_3 의 값(`doc2, doc4, doc5`)만 남습니다. 입력에 `documents` 를 주지 않았으므로 node_2 의 결과는 node_3 에서 **덮어써졌습니다.**

`(2) Reducer를 별도로 지정하는 경우 `
- `Annotated` 사용하여 지정한 reducer 작동 방식에 따라 기존 상태 정보를 업데이트 
- 리스트를 병합하는 `operator.add`를 사용하면, activity_log를 누적하는 방식으로 노드를 구현
- add 함수는 자체로 리스트를 병합하는 동작을 수행하지만, 이 동작이 LangGraph의 상태 병합 규칙으로 통합되려면 Annotated를 통해 해당 타입에 연결되어야 합니다. 

In [ ]:
from operator import add  # 두 리스트를 이어 붙이는 함수 (add(a, b) == a + b)
from typing import Annotated, List, TypedDict
from langgraph.graph import StateGraph, START, END


# Annotated[타입, reducer]: 해당 키의 새 값을 reducer 로 기존 값과 합침 (덮어쓰지 않음)
class ReducerState(TypedDict):
    query: str
    documents: Annotated[List[str], add]


def node_1(state: ReducerState) -> ReducerState:
    """query 를 그대로 반환 (documents 는 건드리지 않음)"""
    print("---Node 1 (query update)---")
    query = state["query"]
    return {"query": query}


def node_2(state: ReducerState) -> ReducerState:
    """1차 문서를 반환 — add reducer 로 기존 documents 뒤에 이어 붙음"""
    print("---Node 2 (add documents)---")
    return {"documents": ["doc1.pdf", "doc2.pdf", "doc3.pdf"]}


def node_3(state: ReducerState) -> ReducerState:
    """2차 문서를 반환 — 마찬가지로 이어 붙음 (중복도 그대로 유지)"""
    print("---Node 3 (add more documents)---")
    return {"documents": ["doc2.pdf", "doc4.pdf", "doc5.pdf"]}


builder = StateGraph(ReducerState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
# 실행하면 documents 는 ["doc1.pdf", "doc2.pdf", "doc3.pdf", "doc2.pdf", "doc4.pdf", "doc5.pdf"] (중복 포함)

In [ ]:
# 초기 상태
initial_state = {"query": "Reducer를 사용한 경우"}

# 그래프 실행 
final_state = graph.invoke(initial_state)

# 최종 상태 출력
print("최종 상태:", final_state)

> **결과 해석**: `documents` 에 node_2 와 node_3 의 결과가 **모두 누적**됩니다. `doc2.pdf` 는 중복으로 두 번 들어 있습니다. 중복을 없애려면 아래 커스텀 reducer 를 사용합니다.

`(3) Custom Reducer 사용 `
- 상태 업데이트가 기본적인 덮어쓰기나 병합만으로 해결되지 않을 때 유용한 방법
- 중복 제거, 최대/최소 값 유지, 조건부 병합 등의 특정 비즈니스 로직이 필요한 경우에 적용
- **커스텀 병합 함수(custom reducer)**를 사용하여 노드 간에 상태를 효율적으로 전달하는 방법을 보여줍니다. 
- Annotated를 사용해 특정 필드(documents)의 병합 방식을 기본 덮어쓰기 방식 대신 직접 정의한 함수로 지정합니다.

In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END


def reduce_unique_documents(left: list | None, right: list | None) -> list:
    """두 문서 리스트를 합치고 중복을 제거합니다. (등장 순서는 유지)

    left: 상태에 이미 있는 값, right: 노드가 새로 반환한 값
    """
    # 첫 호출 시 left 가 None 일 수 있으므로 빈 리스트로 대체
    left = left or []
    right = right or []
    # set 은 순서를 보장하지 않으므로 dict.fromkeys 로 순서를 유지하며 중복 제거
    return list(dict.fromkeys(left + right))


class CustomReducerState(TypedDict):
    query: str
    # documents 에 새 값이 들어올 때마다 reduce_unique_documents(기존, 새값) 결과로 갱신
    documents: Annotated[List[str], reduce_unique_documents]


def node_1(state: CustomReducerState) -> CustomReducerState:
    """query 를 그대로 반환"""
    print("---Node 1 (query update)---")
    query = state["query"]
    return {"query": query}


def node_2(state: CustomReducerState) -> CustomReducerState:
    """1차 문서를 상태에 추가"""
    print("---Node 2 (add documents)---")
    return {"documents": ["doc1.pdf", "doc2.pdf", "doc3.pdf"]}


def node_3(state: CustomReducerState) -> CustomReducerState:
    """2차 문서 추가 — 이미 있는 doc2.pdf 는 reducer 가 제거"""
    print("---Node 3 (add more documents)---")
    return {"documents": ["doc2.pdf", "doc4.pdf", "doc5.pdf"]}


builder = StateGraph(CustomReducerState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()

In [ ]:
# 그래프 구조 확인 (mermaid 코드)
mermaid_code = graph.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

* https://mermaid.live/ 에서  mermaid_code 로 직접 확인한다.

* [Graph이미지](https://mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNpVkcFugzAMhl8FeZdWghacQmha9bI-wk4bU5VCUpAgoBCkdVXffWnoQD3Z_mT_v53cIG8LAQwumnel93HcZSozp1NvuLZh8bXvDlO1X3eH7yVjTFa6N49GZYdP0WIMy4ngSHAmZCRkOcoLVUziLp-kaz4qT6ZeEBy80WA3W84UZ4ozJTMljj59HM6tSX8U0iuE5ENtPFnVNXuTKEMp_bpSIihFdSkNi1b4MuAOd-1B2_G8MlcWvjQ81n_KneU5kTn49mmrApjkdS98aIRu-KOGW6Y8LwNTikZkwGz6XCeDTN3tXMfVZ9s2wIwe7KRuh0v5XwxdwY04Vtz-WzOJa3uj0O_toAwwjKnTAHaDH2DU3kJJiDHZJhHZbGMfrsCiTbpKE0y3GKdkQzG9-_DrTMMVpQQpYpQQDENC0_sffQK3NA)

In [ ]:
# 초기 상태
initial_state = {"query": "채식주의자를 위한 비건 음식을 추천해주세요.", "documents": []}

# 그래프 실행
final_state = graph.invoke(initial_state)

# 최종 상태 출력 (기대값: doc1, doc2, doc3, doc4, doc5 — 중복 없이 순서 유지)
print("최종 상태:", final_state)

#### 2. MessagesState 와 add_messages
* 대화형 앱은 상태로 **메시지 목록(`HumanMessage`, `AIMessage` ...)** 을 가집니다. 노드는 새 메시지만 반환하고, 기존 목록에 **추가**되도록 `add_messages` reducer 를 사용합니다.
* `add_messages` 는 `operator.add` 와 달리 **메시지 ID 로 기존 메시지를 갱신**하고, `("user", "안녕")` 같은 튜플도 메시지 객체로 변환합니다.
* 이 기능을 미리 만들어 둔 상태가 **`MessagesState`** 입니다. (`messages: Annotated[list[AnyMessage], add_messages]` 한 개 키를 가짐)

| 방식 | 코드 | 비고 |
|---|---|---|
| 직접 정의 | `messages: Annotated[list[AnyMessage], add_messages]` | 키 이름·타입을 직접 제어 |
| 미리 만든 상태 | `StateGraph(MessagesState)` | 가장 간단. 키 추가는 상속으로 |

> 이전 버전에서는 `MessageGraph` 를 사용했지만, 지금은 `StateGraph(MessagesState)` 를 사용합니다.

`(1) Messages State 정의`
- 이전 대화 기록을 그래프 상태에 메시지 목록으로 저장하는 것이 유용
- 그래프 상태에 Message 객체 목록을 저장하는 키(채널)를 추가하고, 이 키에 리듀서 함수를 추가 
- 리듀서 함수 선택:
    - operator.add를 사용하면: 새 메시지를 기존 목록에 단순히 추가
    - add_messages 함수를 사용하면:
        - 새 메시지는 기존 목록에 추가
        - 기존 메시지 업데이트도 올바르게 처리 (메시지 ID를 추적)
```python
    class MessageState(TypedDict):
        messages: Annotated[list[AnyMessage], add_messages]        
```    

##### 1) 기본 State 초기화 방법을 사용

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


# 직접 정의: messages 키에 add_messages reducer 연결
class GraphState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

##### 2) MessagesState 상태 객체를 사용

In [ ]:
# 미리 만들어진 MessagesState 를 상속해 필드를 추가하는 방식 (messages 키는 이미 포함됨)
from typing import List
from langchain_core.documents import Document
from langgraph.graph import MessagesState


class GraphState(MessagesState):
    documents: List[Document]
    grade: float
    num_generation: int

`(2) LLM 준비`
* 아래 실습(2단계)에서 LLM 을 호출합니다. API 키는 `.env` 에서 읽는다.**

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 제공자별 API 키 
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

`(3) LLM 설정 (Groq / OpenAI / Upstage 선택)`
- `PROVIDER` 값만 바꾸면 이후 모든 셀의 모델이 바뀝니다. (Groq 는 OpenAI 호환 API 라서 `ChatOpenAI` 에 `base_url` 만 다르게 연결)
- 바꾼 뒤에는 이 셀부터 다시 실행하세요. OpenAI 선택 시 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "upstage"

# 제공자별 설정 (cls: 사용할 LangChain 채팅 모델 클래스)
LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o-mini",  # 테스트에는 작은 모델 사용 (대안: "gpt-4o")
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: 모델 클래스에 그대로 전달할 추가 옵션 (예: seed=42, timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    model_cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = model_cls(**config, temperature=temperature, max_retries=MAX_RETRIES, **kwargs)
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 사용할 LLM 생성 (PROVIDER 설정에 따라 groq / openai / upstage)
llm = get_llm()

##### 기본적인 MessagesState 사용

* `state["messages"]` 는 메시지 객체들의 **리스트**입니다.
    ```python
    state["messages"] = [
        HumanMessage(content="첫 번째 질문"),
        AIMessage(content="첫 번째 답변"),
        HumanMessage(content="두 번째 질문"),
    ]
    ```
* 시스템 메시지 뒤에 대화 기록을 붙일 때는 리스트를 **펼쳐서(`*`)** 넣습니다. (JavaScript 의 spread 연산자와 같은 역할)
    ```python
    [SystemMessage("시스템 메시지"), *state["messages"]]   # 올바름: 평평한 리스트
    [SystemMessage("시스템 메시지"), state["messages"]]    # 잘못됨: 리스트 안에 리스트
    ```
* 메시지 목록을 모델에 넘길 때는 `ChatPromptTemplate` 대신 **메시지 리스트를 `llm.invoke()` 에 직접** 전달합니다. 대화 내용에 `{}` 가 있으면 템플릿 변수로 오해되어 오류가 날 수 있기 때문입니다.

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = "당신은 친절한 AI 어시스턴트입니다. 사용자의 질문에 친절하게 답변해주세요."


def generate_response(state: MessagesState) -> MessagesState:
    """시스템 프롬프트 + 지금까지의 대화로 AI 응답을 만들어 messages 에 추가합니다."""
    response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT), *state["messages"]])
    # add_messages reducer 가 이 응답을 기존 messages 뒤에 추가
    return {"messages": [response]}


# MessagesState 를 상태로 사용하는 그래프: START → generate → END
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("generate", generate_response)
graph_builder.add_edge(START, "generate")
graph_builder.add_edge("generate", END)

app = graph_builder.compile()
app

In [ ]:
# 1턴: 초기 메시지로 그래프 실행 (MessagesState 도 딕셔너리로 초기화)
initial_state = {"messages": [HumanMessage(content="안녕하세요! 오늘 날씨가 어때요?")]}
result = app.invoke(initial_state)

for m in result["messages"]:
    m.pretty_print()  # Human / AI 메시지를 구분해 출력

# 2턴: 이전 대화(Human, AI)에 새 질문을 이어서 전달 — 그래프는 대화를 기억하지 않으므로 직접 넘겨야 함
# (대화를 자동으로 기억하게 하는 방법은 체크포인터(MemorySaver) — ReAct_Memory 노트북 참고)
next_state = {"messages": result["messages"] + [HumanMessage(content="서울 날씨는 어떻게 될까요?")]}
next_result = app.invoke(next_state)
print("두 번째 AI 응답:", next_result["messages"][-1].content)

#### 여러 상태 필드를 사용하는 경우
* `MessagesState` 를 상속해 필드를 추가합니다. 필드별로 reducer 를 다르게 지정할 수 있습니다.
    * `conversation_count: Annotated[int, add]` → 노드가 **증가분(1)** 을 반환하면 합산됩니다.
    * `last_topic: str` → reducer 없음, 마지막 값으로 덮어씁니다.
* `TypedDict` 계열 상태에는 기본값(`= 0`)을 쓸 수 없습니다. 초기값은 `invoke` 입력으로 전달합니다.

In [ ]:
from operator import add
from typing import Annotated
from langgraph.graph import MessagesState


class ExtendedState(MessagesState):
    """MessagesState 에 대화 횟수·마지막 주제 필드를 추가한 확장 상태"""
    conversation_count: Annotated[int, add]  # 대화 횟수 (add reducer: 반환한 증가분이 누적됨)
    last_topic: str  # 마지막 주제 (덮어쓰기)

In [ ]:
def generate_response(state: ExtendedState) -> ExtendedState:
    """주제를 판별해 응답을 생성하고, 대화 횟수(+1)와 주제를 상태에 기록합니다."""
    # 현재 대화 번호 (프롬프트 표시용). 상태 갱신은 아래에서 증가분 1 만 반환
    conversation_count = state.get("conversation_count", 0) + 1

    # 주제 추출 (간단한 키워드 예제)
    last_message = state["messages"][-1].content.lower()
    topic = "날씨" if "날씨" in last_message else "일반"
    print(f"Topic = {topic}")

    system_prompt = f"당신은 친절한 AI 어시스턴트입니다. (대화 #{conversation_count}, 주제: {topic})"
    response = llm.invoke([SystemMessage(content=system_prompt), *state["messages"]])

    return {
        "messages": [response],
        "conversation_count": 1,  # add reducer 가 기존 값에 더함 (현재 값+1 을 반환하면 이중 합산됨)
        "last_topic": topic,
    }

graph_builder = StateGraph(ExtendedState)
graph_builder.add_node("generate", generate_response)
graph_builder.add_edge(START, "generate")
graph_builder.add_edge("generate", END)

app = graph_builder.compile()
app

In [ ]:
# 1턴
query = "VueJS를 만든 사람이 누구인가요?"
initial_state = {
    "messages": [HumanMessage(content=query)],
    "conversation_count": 0,
    "last_topic": "",
}
result = app.invoke(initial_state)
print("대화 횟수:", result["conversation_count"])
print("주제:", result["last_topic"])
print("AI 응답:", result["messages"][-1].content)

# 2턴: 이전 결과의 messages 와 conversation_count 를 넘기면 횟수가 이어서 누적됨 (1 → 2)
result = app.invoke({
    "messages": result["messages"] + [HumanMessage(content="서울 날씨는 어때요?")],
    "conversation_count": result["conversation_count"],
    "last_topic": result["last_topic"],
})
print("대화 횟수:", result["conversation_count"])
print("주제:", result["last_topic"])
print("AI 응답:", result["messages"][-1].content)